# Anomaly Detection Notebook

This notebook demonstrates anomaly detection in government spending transactions using statistical methods.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

# Load datasets
transactions = pd.read_csv('../datasets/raw/transactions.csv')
schemes = pd.read_csv('../datasets/raw/schemes.csv')

print(f"Transactions: {len(transactions)} rows")
print(f"Schemes: {len(schemes)} rows")
transactions.head()

In [ ]:
# Department-wise anomaly detection using Z-score
from statistics import mean, stdev

dept_amounts = {}
for _, row in transactions.iterrows():
    dept_amounts.setdefault(row['department'], []).append(row['amount'])

anomalies = []
for dept, amounts in dept_amounts.items():
    if len(amounts) < 2:
        continue
    avg = mean(amounts)
    sd = stdev(amounts)
    for _, row in transactions.iterrows():
        if row['department'] == dept:
            z = (row['amount'] - avg) / sd if sd > 0 else 0
            severity = 'high' if abs(z) > 2 else 'medium' if abs(z) > 1.5 else 'low'
            if abs(z) > 1.5:
                anomalies.append({
                    'type': 'department_outlier',
                    'department': dept,
                    'transaction_id': row['transaction_id'],
                    'amount': row['amount'],
                    'severity': severity,
                    'z_score': round(z, 2)
                })

anomalies_df = pd.DataFrame(anomalies)
print(f"Detected {len(anomalies_df)} anomalies")
anomalies_df

In [ ]:
# Summary statistics
if not anomalies_df.empty:
    print('Anomaly severity distribution:')
    print(anomalies_df['severity'].value_counts())
    print('\nTop anomalies by Z-score:')
    print(anomalies_df.sort_values('z_score', key=abs, ascending=False).head(10))
else:
    print('No anomalies detected')